In [1]:
import os
import torch

from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from yacs.config import CfgNode as CN

from data.dataset import make_dataset
from eval import evaluate
from src.model import Model
from src.utils import clean_exp_savedir
from src.losses import supervised_loss
import argparse

In [2]:
source_train_loader, _, source_test_loader, target_test_loader = (
    make_dataset(
        source_dataset="office31_amazon",
        target_dataset="office31_dslr",
        img_size=384,
        train_bs=16,
        eval_bs=256,
        num_workers=16,
    )
)

In [3]:
model = Model(
    backbone_type="vit_b_32",
    in_dim=768,
    hidden_dim=256,
    out_dim=31,
    imgsize=384,
    attribute_layers=6,
    patch_size=32,
    attr_net_type="transformer",
)
device = torch.device("cuda")
model = model.to(device)

Loaded pretrained weights.


In [4]:
scaler = GradScaler('cuda')
optimizer = torch.optim.AdamW(
    [
        {
            "params": list(model.classifier_head_src.parameters()),
            "lr": 3e-3,
            "weight_decay": 1e-4,
        },
        {
            "params": list(model.visual_prompt_src.parameters()),
            "lr": 5e-3,
            "weight_decay": 1e-4,
        },
    ]
)

epochs = 30
total_steps = epochs * len(source_train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

In [5]:
os.makedirs("exp", exist_ok=True)
exp_save_dir = os.path.join("exp", "exp_1")
writer = SummaryWriter(exp_save_dir)
best_test_acc = 0
# Training loop
for epoch in range(epochs):
    running_loss = 0.0
    model.train()
    pbar = tqdm(
        source_train_loader,
        total=len(source_train_loader),
        desc=f"Epoch {epoch + 1}",
        ncols=100,
    )

    for batch_idx, source_data in enumerate(pbar):
        pbar.set_description_str(f"Epoch {epoch + 1}", refresh=True)
        current_step = epoch * len(source_train_loader) + batch_idx
        # weak_img, strong_img, label
        weak_img, _, src_labels = source_data 

        weak_img = weak_img.to(device)
        src_labels = src_labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            logit_s = model(weak_img, branch="src")
            loss = supervised_loss(logit_s, src_labels)
            running_loss += loss.item()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        writer.add_scalar("Source/Train BatchLoss", loss.item(), current_step)
        writer.add_scalar(
            "Source/Running loss",
            running_loss / len(source_train_loader),
            current_step,
        )

    test_loss_src, test_accuracy_src = evaluate(
        model, branch="src", test_loader=source_test_loader, device=device
    )
    test_loss_tgt, test_accuracy_tgt = evaluate(
        model, branch="src", test_loader=target_test_loader, device=device
    )
    writer.add_scalar("Source/Test EpochLoss", test_loss_src, epoch)
    writer.add_scalar("Source/Test Accuracy", test_accuracy_src, epoch)

    writer.add_scalar("Target/Test EpochLoss", test_loss_tgt, epoch)
    writer.add_scalar("Target/Test Accuracy", test_accuracy_tgt, epoch)

    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Source: {test_loss_src:.4f}, Test Accuracy Source: {test_accuracy_src:.2f}%"
    )
    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Target: {test_loss_tgt:.4f}, Test Accuracy Target: {test_accuracy_tgt:.2f}%"
    )

    if test_accuracy_src > best_test_acc:
        best_test_acc = test_accuracy_src
        ckpt_path = os.path.join(
            exp_save_dir, f"bi_best_{test_accuracy_src:.2f}.pth"
        )
        torch.save(
            {
                "epoch": epoch,
                "best_test_acc": best_test_acc,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
            },
            ckpt_path,
        )
        print(f"New best checkpoint saved: {ckpt_path}")
        if test_accuracy_src == 100:
            break

Epoch 1: 100%|████████████████████████████████████████████████████| 176/176 [00:21<00:00,  8.20it/s]


Epoch [1/30] Test Loss Source: 0.2701, Test Accuracy Source: 92.65%
Epoch [1/30] Test Loss Target: 0.4111, Test Accuracy Target: 89.18%
New best checkpoint saved: exp/exp_1/bi_best_92.65.pth


Epoch 2: 100%|████████████████████████████████████████████████████| 176/176 [00:20<00:00,  8.68it/s]


Epoch [2/30] Test Loss Source: 0.1390, Test Accuracy Source: 96.17%
Epoch [2/30] Test Loss Target: 0.5118, Test Accuracy Target: 86.97%
New best checkpoint saved: exp/exp_1/bi_best_96.17.pth


Epoch 3: 100%|████████████████████████████████████████████████████| 176/176 [00:22<00:00,  7.96it/s]


...

Epoch [8/30] Test Loss Source: 0.0168, Test Accuracy Source: 99.47%
Epoch [8/30] Test Loss Target: 0.6579, Test Accuracy Target: 82.36%


Epoch 9: 100%|████████████████████████████████████████████████████| 176/176 [00:20<00:00,  8.74it/s]


Epoch [9/30] Test Loss Source: 0.0093, Test Accuracy Source: 99.68%
Epoch [9/30] Test Loss Target: 0.6869, Test Accuracy Target: 82.16%


Epoch 10: 100%|███████████████████████████████████████████████████| 176/176 [00:20<00:00,  8.43it/s]


Epoch [10/30] Test Loss Source: 0.0018, Test Accuracy Source: 100.00%
Epoch [10/30] Test Loss Target: 0.6354, Test Accuracy Target: 85.17%
New best checkpoint saved: exp/exp_1/bi_best_100.00.pth


In [6]:
def evaluate_class_wise(model, branch, test_loader, device, num_classes):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    criterion = torch.nn.CrossEntropyLoss()

    # Initialize tensors to track correct predictions and totals per class
    correct_per_class = torch.zeros(num_classes, device=device)
    total_per_class = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            pred = model(images, branch=branch)
            loss = criterion(pred, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(pred, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Update class-wise metrics for the current batch
            for c in range(num_classes):
                class_mask = (labels == c)
                total_per_class[c] += class_mask.sum()
                correct_per_class[c] += (predicted[class_mask] == labels[class_mask]).sum()

    avg_loss = total_loss / total
    accuracy = 100 * correct / total
    
    # Calculate class-wise accuracy (in percentage) and handle division by zero
    class_wise_accuracy = torch.where(
        total_per_class > 0, 
        (correct_per_class / total_per_class) * 100, 
        torch.tensor(0.0, device=device)
    )

    return avg_loss, accuracy, class_wise_accuracy

In [7]:
_, _, class_wise_acc = evaluate_class_wise(model, branch="src", test_loader=target_test_loader, device=device, num_classes=31)

In [9]:
class_wise_acc

tensor([100.0000, 100.0000, 100.0000,  41.6667,  93.7500,  83.3333, 100.0000,
        100.0000,  73.3333,  80.0000, 100.0000,  80.0000,  83.3333,  93.7500,
         67.7419,  77.2727, 100.0000, 100.0000, 100.0000, 100.0000,  92.3077,
        100.0000,  95.6522,  77.7778,  60.0000, 100.0000, 100.0000,  48.1481,
         57.1429,  90.9091, 100.0000], device='cuda:0')

In [10]:
model_2 = Model(
    backbone_type="vit_b_32",
    in_dim=768,
    hidden_dim=256,
    out_dim=31,
    imgsize=384,
    attribute_layers=6,
    patch_size=32,
    attr_net_type="conv",
)
device = torch.device("cuda")
model_2 = model_2.to(device)

Loaded pretrained weights.


In [11]:
scaler = GradScaler('cuda')
optimizer = torch.optim.AdamW(
    [
        {
            "params": list(model_2.classifier_head_src.parameters()),
            "lr": 3e-3,
            "weight_decay": 1e-4,
        },
        {
            "params": list(model_2.visual_prompt_src.parameters()),
            "lr": 5e-3,
            "weight_decay": 1e-4,
        },
    ]
)

epochs = 30
total_steps = epochs * len(source_train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)

In [12]:
os.makedirs("exp", exist_ok=True)
exp_save_dir = os.path.join("exp", "exp_2")
writer = SummaryWriter(exp_save_dir)
best_test_acc = 0
# Training loop
for epoch in range(epochs):
    running_loss = 0.0
    model_2.train()
    pbar = tqdm(
        source_train_loader,
        total=len(source_train_loader),
        desc=f"Epoch {epoch + 1}",
        ncols=100,
    )

    for batch_idx, source_data in enumerate(pbar):
        pbar.set_description_str(f"Epoch {epoch + 1}", refresh=True)
        current_step = epoch * len(source_train_loader) + batch_idx
        # weak_img, strong_img, label
        weak_img, _, src_labels = source_data 

        weak_img = weak_img.to(device)
        src_labels = src_labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            logit_s = model_2(weak_img, branch="src")
            loss = supervised_loss(logit_s, src_labels)
            running_loss += loss.item()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        writer.add_scalar("Source/Train BatchLoss", loss.item(), current_step)
        writer.add_scalar(
            "Source/Running loss",
            running_loss / len(source_train_loader),
            current_step,
        )

    test_loss_src, test_accuracy_src = evaluate(
        model_2, branch="src", test_loader=source_test_loader, device=device
    )
    test_loss_tgt, test_accuracy_tgt = evaluate(
        model_2, branch="src", test_loader=target_test_loader, device=device
    )
    writer.add_scalar("Source/Test EpochLoss", test_loss_src, epoch)
    writer.add_scalar("Source/Test Accuracy", test_accuracy_src, epoch)

    writer.add_scalar("Target/Test EpochLoss", test_loss_tgt, epoch)
    writer.add_scalar("Target/Test Accuracy", test_accuracy_tgt, epoch)

    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Source: {test_loss_src:.4f}, Test Accuracy Source: {test_accuracy_src:.2f}%"
    )
    print(
        f"Epoch [{epoch + 1}/{epochs}] Test Loss Target: {test_loss_tgt:.4f}, Test Accuracy Target: {test_accuracy_tgt:.2f}%"
    )

    if test_accuracy_src > best_test_acc:
        best_test_acc = test_accuracy_src
        ckpt_path = os.path.join(
            exp_save_dir, f"bi_best_{test_accuracy_src:.2f}.pth"
        )
        torch.save(
            {
                "epoch": epoch,
                "best_test_acc": best_test_acc,
                "model_state_dict": model_2.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
            },
            ckpt_path,
        )
        print(f"New best checkpoint saved: {ckpt_path}")
        if test_accuracy_src == 100:
            break

Epoch 1: 100%|████████████████████████████████████████████████████| 176/176 [00:18<00:00,  9.36it/s]


Epoch [1/30] Test Loss Source: 0.2902, Test Accuracy Source: 91.23%
Epoch [1/30] Test Loss Target: 0.4391, Test Accuracy Target: 87.37%
New best checkpoint saved: exp/exp_2/bi_best_91.23.pth


Epoch 2: 100%|████████████████████████████████████████████████████| 176/176 [00:18<00:00,  9.43it/s]


Epoch [2/30] Test Loss Source: 0.1699, Test Accuracy Source: 95.28%
Epoch [2/30] Test Loss Target: 0.4967, Test Accuracy Target: 85.37%
New best checkpoint saved: exp/exp_2/bi_best_95.28.pth


Epoch 3: 100%|████████████████████████████████████████████████████| 176/176 [00:18<00:00,  9.60it/s]


...

Epoch [16/30] Test Loss Source: 0.0016, Test Accuracy Source: 99.93%
Epoch [16/30] Test Loss Target: 0.7397, Test Accuracy Target: 84.77%


Epoch 17: 100%|███████████████████████████████████████████████████| 176/176 [00:17<00:00, 10.16it/s]


Epoch [17/30] Test Loss Source: 0.0016, Test Accuracy Source: 99.96%
Epoch [17/30] Test Loss Target: 0.6129, Test Accuracy Target: 84.57%
New best checkpoint saved: exp/exp_2/bi_best_99.96.pth


Epoch 18: 100%|███████████████████████████████████████████████████| 176/176 [00:17<00:00,  9.98it/s]


Epoch [18/30] Test Loss Source: 0.0006, Test Accuracy Source: 100.00%
Epoch [18/30] Test Loss Target: 0.5989, Test Accuracy Target: 85.17%
New best checkpoint saved: exp/exp_2/bi_best_100.00.pth


In [13]:
_, _, class_wise_acc_model2 = evaluate_class_wise(model_2, branch="src", test_loader=target_test_loader, device=device, num_classes=31)

In [14]:
class_wise_acc_model2

tensor([100.0000, 100.0000, 100.0000,  75.0000,  81.2500,  66.6667, 100.0000,
        100.0000,  93.3333, 100.0000, 100.0000,  70.0000,  87.5000,  87.5000,
         70.9677,  81.8182, 100.0000, 100.0000, 100.0000,  90.0000,  92.3077,
        100.0000,  91.3044,  66.6667,  20.0000, 100.0000, 100.0000,  74.0741,
         52.3810,  68.1818, 100.0000], device='cuda:0')

In [48]:
class GradCam:
    def __init__(self, model, target):
        self.model = model.eval()
        self.feature = None
        self.gradient = None
        self.target = target
        self._get_hook()

    def _get_features_hook(self, module, input, output):
        self.feature = self.reshape_transform(output)

    def _get_grads_hook(self, module, input, output):
        # Register hook on the output tensor to capture gradients
        def _store_grad(grad):
            self.gradient = self.reshape_transform(grad)
        output.register_hook(_store_grad)

    def _get_hook(self):
        self.target.register_forward_hook(self._get_features_hook)
        self.target.register_forward_hook(self._get_grads_hook)

    def reshape_transform(self, tensor):
        result = tensor[:, 1:, :]                      # remove CLS token [B, N-1, C]
        B, num_patches, C = result.shape
        H = W = int(num_patches ** 0.5)
        if H * W != num_patches:
            raise ValueError(f"Non-square patch grid: {num_patches} patches")
        return result.reshape(B, H, W, C).permute(0, 3, 1, 2)  # [B, C, H, W]

    def __call__(self, inputs):
        self.model.zero_grad()
        output = self.model(inputs, branch="src")
        index = np.argmax(output.cpu().data.numpy(), axis=1)    # [B] - per image

        # Backward on the top class score summed over batch
        target = output[range(len(index)), index].sum()
        target.backward()

        # Generate CAM per image in the batch
        cams = []
        for i in range(inputs.shape[0]):
            gradient = self.gradient[i].cpu().data.numpy()      # [C, H, W]
            weight = np.mean(gradient, axis=(1, 2))             # [C]
            feature = self.feature[i].cpu().data.numpy()        # [C, H, W]
            cam = np.sum(feature * weight[:, np.newaxis, np.newaxis], axis=0)
            cam = np.maximum(cam, 0)
            cam = cam - np.min(cam)
            cam = cam / (np.max(cam) + 1e-8)
            cam = cv2.resize(cam, (inputs.shape[-1], inputs.shape[-2]))
            cams.append(cam)
        return np.stack(cams, axis=0)                           # [B, H, W]

In [49]:
def prepare_input(image):
    image = image.copy() 

    # Normalize the image using the mean and standard deviation
    means = np.array([0.5, 0.5, 0.5])
    stds = np.array([0.5, 0.5, 0.5])
    image -= means
    image /= stds

    # Transpose the image to match the model's expected input format (C, H, W)
    image = np.ascontiguousarray(np.transpose(image, (2, 0, 1)))
    image = image[np.newaxis, ...]  # Add batch dimension

    return torch.tensor(image, requires_grad=True)

def gen_cam(image, mask):
    # Create a heatmap from the Grad-CAM mask
    heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255

    # Superimpose the heatmap on the original image
    cam = (1 - 0.5) * heatmap + 0.5 * image
    cam = cam / np.max(cam)  # Normalize the result
    return np.uint8(255 * cam)  # Convert to 8-bit image
def visualize_grad_cam_vit(model, image):
    target_layer = model.backbone.transformer.blocks[-1].norm1
    grad_cam = GradCam(model, target_layer)
    mask = grad_cam(image)
    results = gen_cam(image, mask)
    return results

In [46]:
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from torchvision.utils import make_grid


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Reverse ImageNet normalization for display. tensor: [C, H, W]"""
    mean = IMAGENET_MEAN.to(tensor.device).view(3, 1, 1)
    std  = IMAGENET_STD.to(tensor.device).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


def evaluate_and_visualize_failures(
    model,
    branch: str,
    test_loader,
    device: torch.device,
    num_classes: int,
    class_names: list[str] | None = None,   # optional human-readable labels
    max_samples_per_class: int = 4,
    save_path: str = "failure_visualization.png",
):
    model.eval()

    criterion = torch.nn.CrossEntropyLoss()
    correct_per_class = torch.zeros(num_classes, device=device)
    total_per_class   = torch.zeros(num_classes, device=device)
    total_loss, total, correct = 0.0, 0, 0

    # Storage: failing_samples[class_idx] = list of (image, mask, pred_idx)
    failing_samples: dict[int, list[tuple]] = {c: [] for c in range(num_classes)}

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            pred = model(images, branch=branch)
            loss = criterion(pred, labels)
            _, predicted = torch.max(pred, 1)

            total_loss += loss.item() * images.size(0)
            total      += labels.size(0)
            correct    += (predicted == labels).sum().item()

            for c in range(num_classes):
                class_mask = (labels == c)
                total_per_class[c]   += class_mask.sum()
                correct_per_class[c] += (predicted[class_mask] == labels[class_mask]).sum()

            wrong_idx = (predicted != labels).nonzero(as_tuple=True)[0]
            if len(wrong_idx) == 0:
                continue
            wrong_images = images[wrong_idx]                    # [M, 3, H, W]
    
    # Re-enable grad just for GradCAM on the misclassified subset
    grad_cam = GradCam(model, model.backbone.transformer.blocks[-1].norm1)
    batch_cams = grad_cam(wrong_images)                     # [M, H, W]
    for local_i, global_idx in enumerate(wrong_idx):
        c = labels[global_idx].item()
        if len(failing_samples[c]) < max_samples_per_class:
            cam_rgb = cv2.applyColorMap(
                np.uint8(255 * batch_cams[local_i]), cv2.COLORMAP_JET
            )                                               # [H, W, 3] BGR
            cam_rgb = cv2.cvtColor(cam_rgb, cv2.COLOR_BGR2RGB)
            cam_tensor = torch.from_numpy(
                cam_rgb.transpose(2, 0, 1)
            ).float() / 255.0                              # [3, H, W]
            failing_samples[c].append((
                images[global_idx].cpu(),
                cam_tensor,
                predicted[global_idx].item(),
            ))

    # ── Identify zero-accuracy classes ───────────────────────────────────────
    class_wise_accuracy = torch.where(
        total_per_class > 0,
        (correct_per_class / total_per_class) * 100,
        torch.tensor(0.0, device=device),
    )
    struggling_classes = sorted(
        [c for c in range(num_classes) if total_per_class[c].item() > 0],
        key=lambda c: class_wise_accuracy[c].item()
    )[:10]

    avg_loss = total_loss / total
    accuracy = 100 * correct / total

    print(f"Overall — Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

    if not struggling_classes:
        print("No zero-accuracy classes found — nothing to visualize.")
        return avg_loss, accuracy, class_wise_accuracy

    # ── Build figure ─────────────────────────────────────────────────────────
    # Layout: one row per failing class, columns = [orig | mask | orig | mask | ...]
    cols_per_sample = 2   # (original, mask) side by side
    n_cols = max_samples_per_class * cols_per_sample
    n_rows = len(struggling_classes)

    fig = plt.figure(figsize=(n_cols * 2.5, n_rows * 3.2))
    fig.suptitle(
        f"Zero-Accuracy Class Failures  ({len(struggling_classes)} classes)",
        fontsize=14, fontweight="bold", y=1.01,
    )

    outer = gridspec.GridSpec(
        n_rows, 1, figure=fig, hspace=0.55
    )

    for row_idx, c in enumerate(struggling_classes):
        class_label = class_names[c] if class_names else f"Class {c}"
        samples     = failing_samples[c]

        inner = gridspec.GridSpecFromSubplotSpec(
            2, max_samples_per_class,
            subplot_spec=outer[row_idx],
            wspace=0.05, hspace=0.15,
        )

        # Row header
        header_ax = fig.add_subplot(outer[row_idx])
        header_ax.set_title(
            f"{class_label}  |  acc= {class_wise_accuracy[c]}%  |  "
            f"{int(total_per_class[c].item())} samples",
            fontsize=10, fontweight="bold", loc="left", pad=12,
            color="crimson",
        )
        header_ax.axis("off")

        for col_idx in range(max_samples_per_class):
            ax_orig = fig.add_subplot(inner[0, col_idx])
            ax_mask = fig.add_subplot(inner[1, col_idx])

            if col_idx < len(samples):
                orig, mask, pred_idx = samples[col_idx]
                pred_label = class_names[pred_idx] if class_names else f"cls {pred_idx}"

                # Original image
                ax_orig.imshow(denormalize(orig).permute(1, 2, 0).numpy())
                ax_orig.set_title(f"orig", fontsize=7, pad=2)

                # Mask — normalise to [0,1] for display
                mask_np = mask.permute(1, 2, 0).numpy()
                mask_np = (mask_np - mask_np.min()) / (mask_np.max() - mask_np.min() + 1e-8)
                ax_mask.imshow(mask_np)
                ax_mask.set_title(f"pred: {pred_label}", fontsize=7, pad=2, color="firebrick")
            else:
                # No sample available for this slot
                ax_orig.text(0.5, 0.5, "—", ha="center", va="center", fontsize=12, color="grey")
                ax_mask.text(0.5, 0.5, "—", ha="center", va="center", fontsize=12, color="grey")

            for ax in (ax_orig, ax_mask):
                ax.axis("off")

        # Row-level sub-labels
        label_ax = fig.add_subplot(inner[:, 0])
        label_ax.set_ylabel("orig\n\nmask", fontsize=8, rotation=0,
                            labelpad=40, va="center", color="dimgrey")
        label_ax.axis("off")

    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save_path}")

    return avg_loss, accuracy, class_wise_accuracy